In [30]:
import pandas as pd
import sqlite3

In [23]:
df = pd.read_csv("googleplaystore.csv")
conn = sqlite3.connect("games.db")
df.to_sql("games", conn, if_exists="replace", index=False)

10841

- "games" --> table name in the database 
- conn --> the database connection to write to
- if_exists="replace" --> if a table with the same name already exists, drop and recreate it
- index=False -->  do not write pandas row index as a column

In [35]:
result = pd.read_sql_query("SELECT * FROM games;", conn)
result

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10836,Sya9a Maroc - FR,FAMILY,4.5,38,53M,"5,000+",Free,0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10837,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3.6M,100+,Free,0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10838,Parkinson Exercices FR,MEDICAL,NaN,3,9.5M,"1,000+",Free,0,Everyone,Medical,"January 20, 2017",1.0,2.2 and up
10839,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,Varies with device,"1,000+",Free,0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device


In [31]:
result = pd.read_sql_query("SELECT App, Genres, Installs FROM games ORDER BY Installs DESC LIMIT 5;", conn)
result

,App,Genres,Installs
0,Life Made WI-Fi Touchscreen Photo Frame,"February 11, 2018",Free
1,Viber Messenger,Communication,"500,000,000+"
2,imo free video calls and chat,Communication,"500,000,000+"
3,Google Duo - High Quality Video Calls,Communication,"500,000,000+"
4,UC Browser - Fast Download Private & Secure,Communication,"500,000,000+"


There seems to be an anomaly in the 'Installs' column. We should identify the incorrect row in the 'games' table.

In [33]:
print(df[df['Installs']== 'Free'].index)

Index([10472], dtype='int64')


The anomaly is located at index 10472. The data in the row is shifted due to a missing column value, but it can be resolved this.

In [39]:
# Shifting values back to their correct columns starting from the second column 
df.iloc[10472, 1:] = df.iloc[10472, 1:].astype(object).shift(1)

# Manually assigning the missing Category value
df.at[10472, 'Category'] = 'PHOTOGRAPHY'

# Syncing the cleaned DataFrame with the SQL database
df.to_sql("games", conn, if_exists="replace", index=False)

10841

In [40]:
result = pd.read_sql_query("SELECT App, Genres, Installs FROM games ORDER BY Installs DESC LIMIT 5;", conn)
result

,App,Genres,Installs
0,Viber Messenger,Communication,"500,000,000+"
1,imo free video calls and chat,Communication,"500,000,000+"
2,Google Duo - High Quality Video Calls,Communication,"500,000,000+"
3,UC Browser - Fast Download Private & Secure,Communication,"500,000,000+"
4,imo free video calls and chat,Communication,"500,000,000+"
